In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')

!pip install mne --quiet

import os
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert, butter, filtfilt

In [ ]:
# === CONFIGURATION ===
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
output_dir = os.path.join(root_dir, 'PAC_Results')
os.makedirs(output_dir, exist_ok=True)

channel_of_interest = 'E6'  # Medial Central
theta_band = (4, 8)
gamma_band = (30, 50)

# === FILTER FUNCTION ===
def bandpass(data, sfreq, band):
    b, a = butter(N=4, Wn=[band[0]/(sfreq/2), band[1]/(sfreq/2)], btype='band')
    return filtfilt(b, a, data)

# === PAC MEASURE FUNCTION ===
def compute_pac(phase_data, amp_data):
    phase = np.angle(hilbert(phase_data))
    amplitude = np.abs(hilbert(amp_data))
    pac = np.abs(np.mean(amplitude * np.exp(1j * phase)))
    return pac

# === LOOP THROUGH PARTICIPANTS ===
pac_values = []
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

for pid in participants:
    epo_path = os.path.join(root_dir, pid, f"{pid}_epo_clean.fif")
    if not os.path.exists(epo_path):
        print(f"[{pid}] ❌ Clean epoch file not found.")
        continue

    try:
        epochs = mne.read_epochs(epo_path, preload=True)
        sfreq = epochs.info['sfreq']

        if channel_of_interest not in epochs.ch_names:
            print(f"[{pid}] Channel {channel_of_interest} not found.")
            continue

        signal = epochs.copy().pick_channels([channel_of_interest]).get_data()
        signal = signal.reshape(signal.shape[0] * signal.shape[2])  # trials × time → flat 1D

        theta = bandpass(signal, sfreq, theta_band)
        gamma = bandpass(signal, sfreq, gamma_band)

        pac = compute_pac(theta, gamma)
        pac_values.append((pid, pac))

        print(f"[{pid}] ✅ PAC: {pac:.4f}")

    except Exception as e:
        print(f"[{pid}] ⚠️ ERROR: {e}")
        continue

# === SAVE TO CSV ===
import pandas as pd
df = pd.DataFrame(pac_values, columns=["Participant", "PAC_Value"])
df.to_csv(os.path.join(output_dir, 'pac_results.csv'), index=False)
print("✅ Saved PAC results to pac_results.csv")


In [ ]:
# Define EEG scalp regions
regions = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37', 'E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31', 'E106', 'E112', 'E80', 'E54', 'E55', 'E79'],
    'Right Central':     ['E87', 'E93', 'E98', 'E102', 'E103', 'E107', 'E108', 'E104', 'E105', 'E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60', 'E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92', 'E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101']
}

# Bandpass filter
def bandpass(data, sfreq, low, high):
    b, a = butter(4, [low / (sfreq / 2), high / (sfreq / 2)], btype='band')
    return filtfilt(b, a, data)

# PAC calculator
def compute_pac(phase_sig, amp_sig):
    phase = np.angle(hilbert(phase_sig))
    amplitude = np.abs(hilbert(amp_sig))
    return np.abs(np.mean(amplitude * np.exp(1j * phase)))

# Set root path to epochs
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

# PAC results container
pac_results = {'Normal': {r: [] for r in regions}, 'Slowed': {r: [] for r in regions}}

for pid in participants:
    epo_file = os.path.join(root_dir, pid, f'{pid}_epo.fif')
    if not os.path.exists(epo_file):
        continue
    try:
        epochs = mne.read_epochs(epo_file, preload=True)
        sfreq = epochs.info['sfreq']

        for cond_label, cond_name in [('Wd2N', 'Normal'), ('Wd2E', 'Slowed')]:
            if cond_label not in epochs.event_id:
                continue
            e = epochs[cond_label].copy().pick_types(eeg=True)

            for region_name, ch_list in regions.items():
                valid_chs = [ch for ch in ch_list if ch in e.ch_names]
                if not valid_chs:
                    continue

                region_data = e.copy().pick_channels(valid_chs).get_data()
                mean_signal = region_data.mean(axis=1).mean(axis=0)

                theta = bandpass(mean_signal, sfreq, 4, 6)
                gamma = bandpass(mean_signal, sfreq, 70, 80)

                pac_val = compute_pac(theta, gamma)
                pac_results[cond_name][region_name].append(pac_val)

    except Exception as err:
        print(f"❌ Error with participant {pid}: {err}")
        continue

# Calculate mean PAC
mean_pac = {
    cond: {region: np.mean(vals) for region, vals in region_dict.items()}
    for cond, region_dict in pac_results.items()
}

# Plot
labels = list(regions.keys())
x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - w/2, [mean_pac['Normal'][r] for r in labels], width=w, label='Normal')
ax.bar(x + w/2, [mean_pac['Slowed'][r] for r in labels], width=w, label='Slowed')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('PAC (Theta Phase → Gamma Amplitude)')
ax.set_title('Phase-Amplitude Coupling (PAC) by Region')
ax.legend()
plt.tight_layout()
plt.show()
